In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import login
from dotenv import load_dotenv
from os import environ
import torch

In [3]:
load_dotenv("../secret.env")
assert environ.get("HF_TOKEN", "").startswith("hf_")
login()

In [ ]:
assert torch.cuda.is_available()
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m-it", device_map="auto")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

device(type='cuda', index=0)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it", device_map="auto")

In [14]:
tokenizer.apply_chat_template([{"role": "user", "content": "What is 4*30?"}], tokenize=False)

'<bos><start_of_turn>user\nWhat is 4*30?<end_of_turn>\n'

In [26]:
# add_generation_prompt adds this <start_of_model> marker
chat_str = [
    tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    for q in ("What is 3*4?", "What is the capital of egypt?")
]
chat_str

['<bos><start_of_turn>user\nWhat is 3*4?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\nWhat is the capital of egypt?<end_of_turn>\n<start_of_turn>model\n']

In [35]:
tokenizer(chat_str, add_special_tokens=False, padding=True, return_offsets_mapping=True, return_tensor="pt")

{'input_ids': [[2, 105, 2364, 107, 3689, 563, 236743, 236800, 236829, 236812, 236881, 106, 107, 105, 4368, 107], [2, 105, 2364, 107, 3689, 563, 506, 5279, 529, 104479, 236881, 106, 107, 105, 4368, 107]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'offset_mapping': [[(0, 5), (5, 20), (20, 24), (24, 25), (25, 29), (29, 32), (32, 33), (33, 34), (34, 35), (35, 36), (36, 37), (37, 50), (50, 51), (51, 66), (66, 71), (71, 72)], [(0, 5), (5, 20), (20, 24), (24, 25), (25, 29), (29, 32), (32, 36), (36, 44), (44, 47), (47, 53), (53, 54), (54, 67), (67, 68), (68, 83), (83, 88), (88, 89)]]}

In [33]:
chat_tokens = tokenizer(chat_str)
print(repr(tokenizer.decode(chat_tokens["input_ids"][0])))

# add_special_tokens=False since we've already applied the chat template
chat_tokens = tokenizer(chat_str, add_special_tokens=False)
print(repr(tokenizer.decode(chat_tokens["input_ids"][0])))

# padding=True pads length of sequences when tokenizing batches
chat_tokens = tokenizer(chat_str, add_special_tokens=False, padding=True, return_offsets_mapping=True, return_tensor="pt")
print(repr(tokenizer.decode(chat_tokens["input_ids"][0])))

'<bos><bos><start_of_turn>user\nWhat is 3*4?<end_of_turn>\n<start_of_turn>model\n'
'<bos><start_of_turn>user\nWhat is 3*4?<end_of_turn>\n<start_of_turn>model\n'
'<bos><start_of_turn>user\nWhat is 3*4?<end_of_turn>\n<start_of_turn>model\n'


In [34]:
offsets = chat_tokens.pop("offset_mapping")
offsets

[[(0, 5),
  (5, 20),
  (20, 24),
  (24, 25),
  (25, 29),
  (29, 32),
  (32, 33),
  (33, 34),
  (34, 35),
  (35, 36),
  (36, 37),
  (37, 50),
  (50, 51),
  (51, 66),
  (66, 71),
  (71, 72)],
 [(0, 5),
  (5, 20),
  (20, 24),
  (24, 25),
  (25, 29),
  (29, 32),
  (32, 36),
  (36, 44),
  (44, 47),
  (47, 53),
  (53, 54),
  (54, 67),
  (67, 68),
  (68, 83),
  (83, 88),
  (88, 89)]]